# Practical 1: Loading and Exploring Unstructured Log Data

**Course:** Programming for Data Science  
**Practical:** 1  
**Dataset:** `cj.log`  
**Student Name:**  
**Enrollment Number:**  
**Date of Experiment:**  

---

## Aim

To load and explore the supplied unstructured security log data, understand its record structure, inspect representative records and identify useful fields for subsequent data analysis and intrusion detection.

## Expected Outcome

After completing this practical, we will be able to:

- Locate and open a large raw log file.
- Read the log without loading the complete file into memory.
- Inspect representative raw records.
- Understand the supplied JSON-array log format.
- Identify useful cybersecurity fields.
- Count valid, blank, concatenated and malformed records.
- Summarize IP addresses, ports, timestamps and user-agents.
- Document fields that are unavailable in the supplied dataset.

## Resources Used

### Software

- Python 3
- Jupyter Notebook
- Visual Studio Code
- Git and GitHub

### Python Libraries

- `pathlib` for filesystem paths
- `json` for parsing JSON records
- `random` for reproducible sampling
- `collections` for efficient counting
- `hashlib` for data-integrity verification
- `pandas` for small summary tables

### Dataset

The experiment uses the supplied `cj.log` security telemetry file. The raw file is kept unchanged in `data/raw`.

In [ ]:
from pathlib import Path
from collections import Counter

import hashlib
import json
import platform
import random
import sys

import pandas as pd

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print("Python version:", sys.version.split()[0])
print("Operating system:", platform.platform())
print("Random seed:", RANDOM_SEED)
print("Current working directory:", Path.cwd())

from pathlib import Path
from collections import Counter
import hashlib
import json
import platform
import random
import sys

import pandas as pd

In [ ]:
def find_project_root(start_path: Path) -> Path:
    """
    Search the current directory and its parents for the project root.

    A valid project root must contain:
    - README.md
    - data/
    - notebooks/
    """
    start_path = start_path.resolve()

    for directory in [start_path, *start_path.parents]:
        has_readme = (directory / "README.md").is_file()
        has_data = (directory / "data").is_dir()
        has_notebooks = (directory / "notebooks").is_dir()

        if has_readme and has_data and has_notebooks:
            return directory

    raise FileNotFoundError(
        "Project root could not be found. "
        "Confirm that README.md, data/ and notebooks/ exist."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

print("Project root:", PROJECT_ROOT)

In [ ]:
LOG_FILE = PROJECT_ROOT / "data" / "raw" / "cj.log"

if not LOG_FILE.is_file():
    raise FileNotFoundError(
        f"Raw log file was not found at:\n{LOG_FILE}"
    )

file_size_bytes = LOG_FILE.stat().st_size
file_size_mib = file_size_bytes / (1024 ** 2)

print("Raw log:", LOG_FILE)
print(f"File size: {file_size_bytes:,} bytes")
print(f"File size: {file_size_mib:.2f} MiB")
print("File available: Yes")

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import platform
import random
import sys

import pandas as pd

## Initial Inspection of Raw Records

Before parsing the complete dataset, we inspect a few physical lines exactly as they appear in the raw file.

This helps us understand:

- The record delimiters
- The number and position of fields
- The representation of missing values
- Whether multiple records can occur on one physical line

In [ ]:
def show_first_nonblank_lines(file_path: Path, number_of_lines: int = 3):
    """Display a limited number of nonblank physical lines."""
    displayed = 0

    with file_path.open("r", encoding="utf-8", errors="replace") as file:
        for line_number, line in enumerate(file, start=1):
            text = line.strip()

            if not text:
                continue

            print(f"Physical line {line_number}:")
            print(text)
            print("-" * 100)

            displayed += 1

            if displayed >= number_of_lines:
                break


show_first_nonblank_lines(LOG_FILE, number_of_lines=3)

## Observed Record Structure

The supplied log uses JSON arrays rather than the conventional Apache combined-log format.

A normal event contains eight positions:

| Position | Index | Proposed Field | Description |
|---:|---:|---|---|
| 1 | 0 | `category_type` | Category or parameter type |
| 2 | 1 | `sub_key` | Associated sub-key |
| 3 | 2 | `timestamp` | Date and time of the event |
| 4 | 3 | `client_ip` | Client/source IP address |
| 5 | 4 | `source_port` | Client-side network port |
| 6 | 5 | `user_agent` | Browser, crawler, scanner or client identity |
| 7 | 6 | `language` | Language preference |
| 8 | 7 | `metadata` | Mixed metadata such as an IP, URL, payload or missing value |

A JSON `null` value becomes Python `None` after parsing.

The supplied records do not directly contain HTTP method, response status, response size or referrer fields.

## Parsing Challenge: Concatenated JSON Arrays

Most physical lines contain one JSON array. Some lines contain multiple arrays placed directly beside one another.

The parser must therefore:

1. Remove surrounding whitespace.
2. Decode one JSON value.
3. Continue from the point where that value ended.
4. Validate that each decoded value is a list.
5. Validate that each event contains exactly eight fields.
6. Record malformed data instead of silently discarding it.

In [ ]:
JSON_DECODER = json.JSONDecoder()
EXPECTED_FIELD_COUNT = 8


def decode_json_values(text: str):
    """
    Decode one or more adjacent JSON values from a string.

    Yields:
        array_position, parsed_value, error_message
    """
    position = 0
    array_position = 0
    text_length = len(text)

    while position < text_length:
        # Skip whitespace between JSON values.
        while position < text_length and text[position].isspace():
            position += 1

        if position >= text_length:
            break

        array_position += 1

        try:
            value, ending_position = JSON_DECODER.raw_decode(text, position)
        except json.JSONDecodeError as error:
            yield array_position, None, str(error)
            break

        yield array_position, value, None
        position = ending_position

In [ ]:
def stream_log_events(file_path: Path):
    """
    Stream events from the raw log while preserving parsing evidence.

    Each yielded dictionary describes either:
    - a valid event,
    - a blank physical line,
    - malformed JSON, or
    - an invalid record structure.
    """
    with file_path.open("r", encoding="utf-8", errors="replace") as file:
        for line_number, line in enumerate(file, start=1):
            text = line.strip()

            if not text:
                yield {
                    "source_line": line_number,
                    "array_position": None,
                    "parse_status": "blank",
                    "record": None,
                    "error": None,
                }
                continue

            for array_position, value, error in decode_json_values(text):
                if error is not None:
                    yield {
                        "source_line": line_number,
                        "array_position": array_position,
                        "parse_status": "invalid_json",
                        "record": None,
                        "error": error,
                    }
                    continue

                if not isinstance(value, list):
                    yield {
                        "source_line": line_number,
                        "array_position": array_position,
                        "parse_status": "not_an_array",
                        "record": value,
                        "error": "Decoded JSON value is not an array.",
                    }
                    continue

                if len(value) != EXPECTED_FIELD_COUNT:
                    yield {
                        "source_line": line_number,
                        "array_position": array_position,
                        "parse_status": "invalid_field_count",
                        "record": value,
                        "error": (
                            f"Expected {EXPECTED_FIELD_COUNT} fields, "
                            f"but found {len(value)}."
                        ),
                    }
                    continue

                yield {
                    "source_line": line_number,
                    "array_position": array_position,
                    "parse_status": "valid",
                    "record": value,
                    "error": None,
                }

In [ ]:
FIELD_NAMES = [
    "category_type",
    "sub_key",
    "timestamp",
    "client_ip",
    "source_port",
    "user_agent",
    "language",
    "metadata",
]


sample_events = []

for event in stream_log_events(LOG_FILE):
    if event["parse_status"] == "valid":
        row = {
            "source_line": event["source_line"],
            "array_position": event["array_position"],
        }

        row.update(dict(zip(FIELD_NAMES, event["record"])))
        sample_events.append(row)

    if len(sample_events) == 5:
        break


sample_dataframe = pd.DataFrame(sample_events)
sample_dataframe

## Complete Dataset Profiling

The raw dataset is too large to load repeatedly for individual calculations. We therefore perform one streaming pass and calculate all major statistics together.

The profiling operation measures:

- Physical lines
- Valid and invalid events
- Blank lines
- Concatenated records
- Missing values
- Timestamp range
- IP-address validity
- Source-port validity
- Unique and frequent values
- A reproducible random sample

This approach is memory-efficient because individual records are processed and then discarded.

from datetime import datetime
import ipaddress
import time

In [ ]:
from datetime import datetime
import ipaddress
import time

## Reservoir Sampling

Ordinary random sampling normally requires the complete dataset to be loaded into memory. Reservoir sampling instead maintains a fixed-size random sample while reading a stream.

For a reservoir of size \(k\), the \(n\)-th record is retained with probability:

\[
P(\text{selected}) = \frac{k}{n}
\]

This allows representative sampling from datasets larger than available memory.

In [ ]:
def is_missing(value) -> bool:
    """Return True when a field is null, empty or whitespace-only."""
    if value is None:
        return True

    if isinstance(value, str) and not value.strip():
        return True

    return False


def profile_log_file(file_path: Path, sample_size: int = 10):
    """
    Profile the complete log in one streaming pass.

    Returns a dictionary containing quality, frequency,
    validation and sampling results.
    """
    start_time = time.perf_counter()
    random.seed(RANDOM_SEED)

    status_counts = Counter()
    missing_counts = Counter()

    ip_counts = Counter()
    user_agent_counts = Counter()
    port_counts = Counter()
    language_counts = Counter()
    category_counts = Counter()
    sub_key_counts = Counter()

    unique_ip_types = {}
    invalid_ip_count = 0
    invalid_port_count = 0
    invalid_timestamp_count = 0

    earliest_timestamp = None
    latest_timestamp = None

    reservoir = []
    valid_records_seen = 0

    physical_line_count = 0
    concatenated_line_count = 0

    current_source_line = None
    items_on_current_line = 0

    for event in stream_log_events(file_path):
        source_line = event["source_line"]
        physical_line_count = max(physical_line_count, source_line)

        # A change in source line means the previous physical line is complete.
        if current_source_line is None:
            current_source_line = source_line

        elif source_line != current_source_line:
            if items_on_current_line > 1:
                concatenated_line_count += 1

            current_source_line = source_line
            items_on_current_line = 0

        status = event["parse_status"]
        status_counts[status] += 1

        if status != "blank":
            items_on_current_line += 1

        if status != "valid":
            continue

        valid_records_seen += 1
        record = event["record"]

        row = dict(zip(FIELD_NAMES, record))
        row["source_line"] = source_line
        row["array_position"] = event["array_position"]

        # -------------------------
        # Reservoir sampling
        # -------------------------
        if len(reservoir) < sample_size:
            reservoir.append(row.copy())
        else:
            replacement_position = random.randint(1, valid_records_seen)

            if replacement_position <= sample_size:
                reservoir[replacement_position - 1] = row.copy()

        # -------------------------
        # Missing values
        # -------------------------
        for field_name, value in zip(FIELD_NAMES, record):
            if is_missing(value):
                missing_counts[field_name] += 1

        (
            category_type,
            sub_key,
            timestamp_text,
            client_ip,
            source_port,
            user_agent,
            language,
            metadata,
        ) = record

        # -------------------------
        # Frequency counters
        # -------------------------
        if not is_missing(category_type):
            category_counts[str(category_type)] += 1

        if not is_missing(sub_key):
            sub_key_counts[str(sub_key)] += 1

        if not is_missing(client_ip):
            ip_counts[str(client_ip)] += 1

        if not is_missing(source_port):
            port_counts[str(source_port)] += 1

        if not is_missing(user_agent):
            user_agent_counts[str(user_agent)] += 1

        if not is_missing(language):
            language_counts[str(language)] += 1

        # -------------------------
        # Timestamp validation
        # -------------------------
        if is_missing(timestamp_text):
            invalid_timestamp_count += 1
        else:
            try:
                parsed_timestamp = datetime.fromisoformat(
                    str(timestamp_text)
                )

                if (
                    earliest_timestamp is None
                    or parsed_timestamp < earliest_timestamp
                ):
                    earliest_timestamp = parsed_timestamp

                if (
                    latest_timestamp is None
                    or parsed_timestamp > latest_timestamp
                ):
                    latest_timestamp = parsed_timestamp

            except (TypeError, ValueError):
                invalid_timestamp_count += 1

        # -------------------------
        # IP-address validation
        # -------------------------
        if not is_missing(client_ip):
            ip_text = str(client_ip)

            if ip_text not in unique_ip_types:
                try:
                    address = ipaddress.ip_address(ip_text)

                    if address.is_loopback:
                        address_type = "loopback"
                    elif address.is_private:
                        address_type = "private"
                    elif address.is_global:
                        address_type = "public"
                    else:
                        address_type = "other"

                    unique_ip_types[ip_text] = address_type

                except ValueError:
                    unique_ip_types[ip_text] = "invalid"
                    invalid_ip_count += 1

        # -------------------------
        # Source-port validation
        # -------------------------
        if not is_missing(source_port):
            try:
                port_number = int(source_port)

                if not 1 <= port_number <= 65535:
                    invalid_port_count += 1

            except (TypeError, ValueError):
                invalid_port_count += 1

    # Finalize the final physical line.
    if items_on_current_line > 1:
        concatenated_line_count += 1

    elapsed_seconds = time.perf_counter() - start_time

    return {
        "physical_line_count": physical_line_count,
        "status_counts": status_counts,
        "concatenated_line_count": concatenated_line_count,
        "valid_record_count": valid_records_seen,
        "missing_counts": missing_counts,
        "ip_counts": ip_counts,
        "user_agent_counts": user_agent_counts,
        "port_counts": port_counts,
        "language_counts": language_counts,
        "category_counts": category_counts,
        "sub_key_counts": sub_key_counts,
        "unique_ip_types": unique_ip_types,
        "invalid_ip_count": invalid_ip_count,
        "invalid_port_count": invalid_port_count,
        "invalid_timestamp_count": invalid_timestamp_count,
        "earliest_timestamp": earliest_timestamp,
        "latest_timestamp": latest_timestamp,
        "reservoir_sample": reservoir,
        "elapsed_seconds": elapsed_seconds,
    }

In [ ]:
profile = profile_log_file(LOG_FILE, sample_size=10)

print("Profiling completed.")
print(f"Processing time: {profile['elapsed_seconds']:.2f} seconds")

In [ ]:
status_counts = profile["status_counts"]

overview_rows = [
    {
        "Metric": "Raw file size",
        "Value": f"{file_size_mib:.2f} MiB",
    },
    {
        "Metric": "Physical lines",
        "Value": f"{profile['physical_line_count']:,}",
    },
    {
        "Metric": "Valid records",
        "Value": f"{profile['valid_record_count']:,}",
    },
    {
        "Metric": "Blank physical lines",
        "Value": f"{status_counts.get('blank', 0):,}",
    },
    {
        "Metric": "Invalid JSON values",
        "Value": f"{status_counts.get('invalid_json', 0):,}",
    },
    {
        "Metric": "Invalid field-count records",
        "Value": f"{status_counts.get('invalid_field_count', 0):,}",
    },
    {
        "Metric": "Non-array JSON values",
        "Value": f"{status_counts.get('not_an_array', 0):,}",
    },
    {
        "Metric": "Lines containing multiple records",
        "Value": f"{profile['concatenated_line_count']:,}",
    },
    {
        "Metric": "Unique IP addresses",
        "Value": f"{len(profile['ip_counts']):,}",
    },
    {
        "Metric": "Unique source ports",
        "Value": f"{len(profile['port_counts']):,}",
    },
    {
        "Metric": "Unique user-agents",
        "Value": f"{len(profile['user_agent_counts']):,}",
    },
    {
        "Metric": "Unique language values",
        "Value": f"{len(profile['language_counts']):,}",
    },
    {
        "Metric": "Earliest timestamp",
        "Value": str(profile["earliest_timestamp"]),
    },
    {
        "Metric": "Latest timestamp",
        "Value": str(profile["latest_timestamp"]),
    },
    {
        "Metric": "Processing time",
        "Value": f"{profile['elapsed_seconds']:.2f} seconds",
    },
]

overview_dataframe = pd.DataFrame(overview_rows)
overview_dataframe

In [ ]:
parsing_status_dataframe = (
    pd.DataFrame(
        profile["status_counts"].items(),
        columns=["Parse Status", "Count"],
    )
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

parsing_status_dataframe

In [ ]:
valid_record_count = profile["valid_record_count"]

missing_rows = []

for field_name in FIELD_NAMES:
    missing_count = profile["missing_counts"].get(field_name, 0)
    missing_percentage = (
        missing_count / valid_record_count * 100
        if valid_record_count
        else 0
    )

    missing_rows.append(
        {
            "Field": field_name,
            "Missing Count": missing_count,
            "Missing Percentage": missing_percentage,
        }
    )

missing_dataframe = pd.DataFrame(missing_rows)
missing_dataframe["Missing Count"] = missing_dataframe[
    "Missing Count"
].map(lambda value: f"{value:,}")

missing_dataframe["Missing Percentage"] = (
    missing_dataframe["Missing Percentage"]
    .map(lambda value: f"{value:.2f}%")
)

missing_dataframe

In [ ]:
reservoir_dataframe = pd.DataFrame(profile["reservoir_sample"])

column_order = [
    "source_line",
    "array_position",
    *FIELD_NAMES,
]

reservoir_dataframe = reservoir_dataframe[column_order]
reservoir_dataframe

## Frequency and Concentration Analysis

Frequency analysis identifies values that dominate the dataset.

In intrusion-detection data, concentration is important because:

- A small number of scanner tools may generate most events.
- A random train/test split may contain the same tools and IPs on both sides.
- High model accuracy may therefore result from memorization rather than generalization.

In [ ]:
def create_frequency_table(
    counter: Counter,
    total_records: int,
    column_name: str,
    top_n: int = 10,
):
    """Create a frequency table with share and cumulative coverage."""
    nonmissing_records = sum(counter.values())
    rows = []
    cumulative_count = 0

    for value, count in counter.most_common(top_n):
        cumulative_count += count

        rows.append(
            {
                column_name: value,
                "Count": count,
                "Share of Valid Records (%)": (
                    count / total_records * 100
                ),
                "Share of Nonmissing Records (%)": (
                    count / nonmissing_records * 100
                    if nonmissing_records
                    else 0
                ),
                "Cumulative Coverage (%)": (
                    cumulative_count / nonmissing_records * 100
                    if nonmissing_records
                    else 0
                ),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
top_user_agents = create_frequency_table(
    counter=profile["user_agent_counts"],
    total_records=profile["valid_record_count"],
    column_name="User Agent",
    top_n=10,
)

top_user_agents

In [ ]:
def pseudonymize_ip(ip_text: str) -> str:
    """
    Convert an IP into a stable display identifier.

    This hides the direct value in notebook outputs but is not
    intended to be cryptographically irreversible anonymization.
    """
    digest = hashlib.sha256(ip_text.encode("utf-8")).hexdigest()
    return f"IP-{digest[:12]}"


top_ip_rows = []

for ip_address, count in profile["ip_counts"].most_common(10):
    top_ip_rows.append(
        {
            "IP Identifier": pseudonymize_ip(ip_address),
            "Count": count,
            "Share of Valid Records (%)": (
                count / profile["valid_record_count"] * 100
            ),
        }
    )

top_ips = pd.DataFrame(top_ip_rows)

top_ips

In [ ]:
top_categories = create_frequency_table(
    counter=profile["category_counts"],
    total_records=profile["valid_record_count"],
    column_name="Category Type",
    top_n=15,
)

top_categories

In [ ]:
top_languages = create_frequency_table(
    counter=profile["language_counts"],
    total_records=profile["valid_record_count"],
    column_name="Language",
    top_n=10,
)

top_languages

In [ ]:
def concentration_summary(counter: Counter, name: str):
    total = sum(counter.values())

    top_1_count = sum(
        count for _, count in counter.most_common(1)
    )
    top_5_count = sum(
        count for _, count in counter.most_common(5)
    )
    top_10_count = sum(
        count for _, count in counter.most_common(10)
    )

    return {
        "Feature": name,
        "Unique Values": len(counter),
        "Top 1 Coverage (%)": (
            top_1_count / total * 100 if total else 0
        ),
        "Top 5 Coverage (%)": (
            top_5_count / total * 100 if total else 0
        ),
        "Top 10 Coverage (%)": (
            top_10_count / total * 100 if total else 0
        ),
    }


concentration_dataframe = pd.DataFrame(
    [
        concentration_summary(
            profile["ip_counts"],
            "Client IP",
        ),
        concentration_summary(
            profile["user_agent_counts"],
            "User Agent",
        ),
        concentration_summary(
            profile["port_counts"],
            "Source Port",
        ),
        concentration_summary(
            profile["language_counts"],
            "Language",
        ),
        concentration_summary(
            profile["category_counts"],
            "Category Type",
        ),
        concentration_summary(
            profile["sub_key_counts"],
            "Sub-key",
        ),
    ]
)

concentration_dataframe


## Dataset Integrity

A cryptographic hash acts as a fingerprint of the raw file. Even a one-character change produces a different SHA-256 value.

Recording this fingerprint allows another researcher to verify that they used the same dataset.

In [ ]:
def calculate_sha256(
    file_path: Path,
    block_size: int = 1024 * 1024,
) -> str:
    """Calculate a SHA-256 fingerprint without loading the file at once."""
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        while block := file.read(block_size):
            sha256.update(block)

    return sha256.hexdigest()


dataset_sha256 = calculate_sha256(LOG_FILE)

print("Dataset:", LOG_FILE.name)
print("Size:", f"{file_size_bytes:,} bytes")
print("SHA-256:", dataset_sha256)

## Interpretation of Frequency and Concentration

The dataset is highly concentrated around a small number of entities and user-agents.

- The most frequent user-agent, `gobuster/3.6`, represents 68.54% of all nonmissing user-agent records.
- Gobuster and DirBuster together represent 88.09% of nonmissing user-agent records.
- The top five user-agents cover 93.16% of nonmissing user-agent activity.
- The most frequent IP represents 29.68% of IP-based events, while the top five IPs cover 76.17%.
- Source ports are highly dispersed: the ten most frequent ports represent only 0.25% of nonmissing port records.
- Category and sub-key values are less concentrated than user-agents, but these fields have extensive missingness.

These results indicate that the dataset is dominated by directory-enumeration activity from a relatively small set of sources. A future machine-learning model could obtain misleadingly high performance by memorizing dominant user-agents or IP identities.

Source port alone appears to be a weak indicator because no small group of ports dominates the dataset. It may become useful when combined with event rate, IP behaviour and temporal features.

## Results and Findings

The raw file contains 2,062,365 physical lines. Of these, 934 are blank, leaving 2,061,431 nonblank physical lines.

The parser recovered 2,062,361 valid eight-field records. The number of valid records is greater than the number of nonblank lines because 911 physical lines contain multiple concatenated JSON arrays. These lines contributed 930 additional records, indicating that some physical lines contain more than two arrays.

No malformed JSON records were detected, and all recovered events followed the expected eight-field structure.

The dataset spans from 8 January 2023 to 19 February 2024 and contains 16,680 unique client IP addresses, 41,884 unique source ports and 5,953 unique user-agents.

The traffic is highly concentrated. The most frequent user-agent, `gobuster/3.6`, represents 68.54% of nonmissing user-agent records. Gobuster and DirBuster together represent 88.09%. This suggests that directory-enumeration activity is a dominant behaviour in the dataset.

A small number of IP addresses also generate most events, while source ports are highly dispersed. Therefore, IP identity and user-agent identity must not be allowed to leak across future training and testing datasets.

The category and sub-key fields contain mixed information, including parameter names and command-like payloads. These fields require semantic separation before feature engineering.

The supplied format does not directly provide HTTP method, status code, response bytes or referrer. Consequently, the data supports detection of observed activity and attack attempts, but it cannot independently confirm successful exploitation.

## Conclusion

The supplied `cj.log` file was loaded and explored successfully using a memory-efficient streaming approach.

A custom JSON decoder was required because some physical lines contain multiple adjacent records. The profiling process validated 2,062,361 structured events without loading the complete dataset into memory.

The analysis identified timestamps, client IPs, source ports, user-agents, language values, categories, sub-keys and metadata as the available fields. It also revealed substantial missingness, highly concentrated scanner activity and mixed-content categorical fields.

The dataset is suitable for behavioural security analysis and attack-attempt detection. However, careful preprocessing, evidence-based labeling and leakage-resistant evaluation will be required before machine learning.